# Notebook 04 — Retrieval exploratoire (Cycle 4)

Sanity check des outputs des scripts `src/retrieval/01..04_*.py` :

- **4.1** FAISS bench (Flat IP / HNSW / IVF_PQ) : recall@k vs latence.
- **4.2** Multi-view RRF (Cormack 2009) : fusion text + vision quand vision dispo.
- **4.3** OOD guardrails : calibration seuil F0.5.
- **4.3** Akinator backend : taux de déclenchement + attributs discriminants.

**Prereqs** :
- Cycle 2.3 Arctic Embed terminé (`data/embeddings/text/text_arctic_*.npy`)
- Cycle 4.1+ scripts exécutés (`reports/05_retrieval/*.json`, `data/index/*.index`)

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl

from src.config import DATA_EMBEDDINGS, DATA_INDEX, DATA_PROCESSED_PRODUCTS, REPORTS_RETRIEVAL, SEED


## 1. Lecture des reports

In [ ]:
for fname in ("faiss_bench.json", "multiview_rrf_eval.json", "ood_calibration.json", "akinator_backend_eval.json"):
    p = REPORTS_RETRIEVAL / fname
    if p.exists():
        print(f"=== {fname} ===")
        print(json.dumps(json.loads(p.read_text(encoding='utf-8')), indent=2, ensure_ascii=False)[:1500])
        print()
    else:
        print(f"⚠️ {fname} absent — relance le script src/retrieval/0X_*.py")


## 2. Sanity check : 5 voisins d'une query random

In [ ]:
import faiss

index = faiss.read_index(str(DATA_INDEX / 'text_arctic_hnsw.index'))
index.hnsw.efSearch = 64

val_emb = np.load(DATA_EMBEDDINGS / 'text' / 'text_arctic_val.npy').astype(np.float32)
train_meta = pl.read_parquet(DATA_PROCESSED_PRODUCTS / 'train.parquet', columns=['parent_asin', 'title', 'brand', '_source_category'])
val_meta = pl.read_parquet(DATA_PROCESSED_PRODUCTS / 'val.parquet', columns=['parent_asin', 'title', 'brand', '_source_category'])

rng = np.random.default_rng(SEED)
q_idx = int(rng.integers(0, val_emb.shape[0]))
query_meta = val_meta.row(q_idx, named=True)
print('QUERY :', query_meta)

_, top5 = index.search(val_emb[q_idx:q_idx+1], 5)
for rank, i in enumerate(top5[0]):
    n = train_meta.row(int(i), named=True)
    print(f"  #{rank+1} {n.get('_source_category', '')[:25]:25} | {n.get('brand', '')[:20]:20} | {n.get('title', '')[:60]}")


## 3. Décision Cycle 4 (lecture humaine attendue)

À documenter ici après lecture des bench :
- Index FAISS retenu pour serving (HNSW si recall ≥ 0.95 + latence < 10 ms)
- Seuil OOD final (best F0.5)
- Taux de déclenchement Akinator (cible : < 20 % des queries)